# Full mouse analysis — single-animal training progression

`Mouse` gathers **every session of one animal** for a paradigm, concatenates the runs onto a
session clock, and builds two tables:

- **`mouse.data.cumul_data`** — the tidy per-**trial** table across all sessions (one row / trial,
  with `session_no`, `session_type`, cumulative `cumul_trial_no`).
- **`mouse.data.summary_data`** — one row per **session** (date, level, task, sf/tf, stats, rig…).

Then we plot with **`piepy.viz`** (the new behaviz-backed layer). Each plot is a plain function
running the same three steps: **group & aggregate → statistical test/fit → plot + significance.**

> **Heads-up (side effects):** building a session with `load_flag` re-parses and **re-saves** to
> your analysis dir, and `mouse.save()` **deletes** the previous saved data. This notebook is
> shipped *unexecuted* — run the cells yourself against your data.

In [ ]:
from piepy.core.mouse import Mouse
import polars as pl

## 1 · Gather the training progression

`load_type` options: `"no_load"` (re-parse every session from scratch), `"reanalyze"`,
`"load_and_add"` (load saved data, analyze only new sessions), `"last_saved"` (just load).
`dateinterval` is optional: `"231124"` (from date → today) or `["231101", "231130"]`.

In [ ]:
mouse = Mouse(animalid="KC304", paradigm="detection", dateinterval=None)
mouse.gather_data(load_type="no_load")

### Per-session summary

In [ ]:
mouse.data.summary_data

### Per-trial cumulative table (the input to every plot)

In [ ]:
cumul = mouse.data.cumul_data
print(cumul.shape)
cumul.head()

### (optional) persist the gathered analysis
Writes `<paradigm>BehaviorData.parquet` + `…Summary.csv` and **removes the previous** saved copy.

In [ ]:
# mouse.save()  # uncomment to save

## 2 · Plot with `piepy.viz` (behaviz)

Each plot is a **one-liner function** returning a `PlotResult` with `.data` (the aggregated
estimates), `.stats` (the test/fit), and `.figure` (the behaviz figure). Steps 1–2 (aggregate +
stats/fit) are wired against `piepy.stats` / `piepy.fitting`; **step 3 — the behaviz drawing — is
the part you fill in**, so `.figure` is `None` until then while `.data`/`.stats` are already there.

Call a plot directly, or by name via the registry. Discover what a paradigm offers:

In [ ]:
from piepy.viz import detection_psychometric, plot, available_plots

available_plots("detection")

**Direct function call** — the simplest one-liner:

In [ ]:
result = detection_psychometric(mouse.data.cumul_data)
result.data        # step 1: hit rate + Wilson CI + n, per condition

In [ ]:
result.stats       # step 2: fitted psychometric (FitResult) per opto condition
# result.figure    # step 3: None until you fill in the behaviz drawing

**By name** via the registry dispatcher — equivalent, handy for loops/dashboards:

In [ ]:
plot("detection", "psychometric", mouse.data.cumul_data).data

### Colours
`piepy.viz.colors` bridges the (old) per-task colour infra so your plotting step can fill
behaviz's `color=` kwargs. It is failure-tolerant — returns `None`/`{}` when the colour config is
absent, so plots fall back to behaviz defaults. (This infra is slated to move into behaviz later.)

In [ ]:
from piepy.viz.colors import get_palette, outcome_colors

pal = get_palette("detection")          # a legacy Color palette, or None
outcome_colors(pal, ["hit", "miss"])    # {'hit': '#..', 'miss': '#..'} or {}

### Filling in a plot
Each plot is a function in `piepy/viz/<paradigm>/<name>.py`. Steps 1–2 are done; the step-3 block
(marked `FILL IN`) is where you draw with behaviz (imported lazily inside the function) and set
`figure`. Once filled, the same one-liner returns the figure too:

```python
result = detection_psychometric(mouse.data.cumul_data, color=pal)
result.figure   # the behaviz figure
```